# Guidance Discovery corpus inspector

Loads the gitignored run artifacts in `output/`:

- `search_api.json` — stage 1 metadata (currently ~16.2k)
- `with_body.json` — stage 2, same rows plus `body_text` (currently ~16.1k)

```bash
cd guidance-discovery
uv sync --group notebook
uv run jupyter notebook inspect_corpus.ipynb
```

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 50)

ROOT = Path(".") if Path("output/search_api.json").exists() else Path("guidance-discovery")
SEARCH_PATH = ROOT / "output" / "search_api.json"
BODY_PATH = ROOT / "output" / "with_body.json"
print(SEARCH_PATH.resolve())
print("search_api exists:", SEARCH_PATH.exists(), "with_body exists:", BODY_PATH.exists())

/Users/bram/Code/defra/ai-eu-trade-accelerator/guidance-discovery/output/search_api.json
search_api exists: True with_body exists: True


In [2]:
def flatten(pages: list[dict]) -> pd.DataFrame:
    rows = []
    for p in pages:
        m = p["meta_data"]
        body = m.get("body_text")
        rows.append(
            {
                "url": p["url"],
                "content_id": p["content_id"],
                "title": m.get("title") or "",
                "description": m.get("description") or "",
                "updated_at": pd.to_datetime(m.get("updated_at"), utc=True, errors="coerce"),
                "document_type": m.get("document_type") or "",
                "view_count": m.get("view_count") or 0,
                "has_body_field": "body_text" in m,
                "has_body": bool(body),
                "body_len": len(body) if body else 0,
                "body_text": body,
            }
        )
    return pd.DataFrame(rows)


search = flatten(json.loads(SEARCH_PATH.read_text()))
print(f"search_api.json  {len(search):,} rows")

if BODY_PATH.exists():
    bodies = flatten(json.loads(BODY_PATH.read_text()))
    print(f"with_body.json   {len(bodies):,} rows  (has_body={int(bodies.has_body.sum()):,}  empty/null={int((~bodies.has_body).sum()):,})")
    df = search.drop(columns=["has_body_field", "has_body", "body_len", "body_text"]).merge(
        bodies[["url", "has_body", "body_len", "body_text"]],
        on="url",
        how="left",
    )
    df["has_body"] = df["has_body"].fillna(False).astype(bool)
    df["body_len"] = df["body_len"].fillna(0).astype(int)
    df["in_with_body"] = df["url"].isin(bodies["url"])
else:
    df = search.copy()
    df["in_with_body"] = False
    print("no with_body.json — metadata only")

print(f"merged           {len(df):,} rows  (discovery-only={int((~df.in_with_body).sum()):,})")
df.head(3)

search_api.json  16,232 rows


with_body.json   16,158 rows  (has_body=15,796  empty/null=362)
merged           16,232 rows  (discovery-only=239)


,url,content_id,title,description,updated_at,document_type,view_count,has_body,body_len,body_text,in_with_body
0,https://www.gov.uk/log-in-register-hmrc-online-services,632d1ae0-0340-4a23-87b0-595bb596e5f4,HMRC online services: sign in or set up an account,"Sign in or set up a personal or business tax account, Self Assessment, Corporation Tax, PAYE for employers, VAT and ...",2024-10-21 14:33:49+00:00,guide,1501621,True,3027,"Sign in to your HM Revenue and Customs (HMRC) online account for things like a personal or business tax account, Sel...",True
1,https://www.gov.uk/tax-free-childcare,6e2a4012-2448-47fd-b7ec-a47396e4b114,Tax-Free Childcare,"What Tax-Free Childcare is, who can get it and how to apply.",2024-11-08 18:10:18+00:00,guide,140516,True,12975,"Tax-Free Childcare can help with the cost of paying for childcare.\n\nYou’ll need to set up a childcare account, whi...",True
2,https://www.gov.uk/guidance/download-the-hmrc-app,444710db-cec0-42aa-8b6b-d23d173c73ec,Download the HMRC app,"How to download and access the HMRC app on a mobile device to get information about your tax, National Insurance and...",2026-08-14 08:22:06+00:00,detailed_guide,91655,True,2264,Watch our YouTube video to find out what you can use the HMRC app for:\n\nMake life easier with the HMRC app.\n\nGet...,True


## Overview

In [3]:
display(Markdown(
    f"""
| | |
|---|---:|
| discovery rows | {len(df):,} |
| in `with_body.json` | {int(df.in_with_body.sum()):,} |
| with non-empty `body_text` | {int(df.has_body.sum()):,} |
| missing / empty body | {int((df.in_with_body & ~df.has_body).sum()):,} |
| document types | {df.document_type.nunique()} |
"""
))

types = (
    df.groupby("document_type")
    .agg(n=("url", "size"), with_body=("has_body", "sum"), median_body_len=("body_len", "median"))
    .sort_values("n", ascending=False)
)
types


| | |
|---|---:|
| discovery rows | 16,232 |
| in `with_body.json` | 15,993 |
| with non-empty `body_text` | 15,634 |
| missing / empty body | 359 |
| document types | 9 |


,n,with_body,median_body_len
document_type,,,
guidance,7068,6970,303.0
detailed_guide,4526,4433,4729.5
document_collection,1785,1427,336.0
manual_section,1377,1361,6467.0
statutory_guidance,642,634,452.0
guide,372,371,5869.5
farming_grant,203,203,9579.0
countryside_stewardship_grant,152,151,4148.5
manual,107,84,233.0


## Filter

Edit `QUERY` and re-run. Matches title, description, url, document_type (case-insensitive). Set `DOCUMENT_TYPE` to pin a type, or `None` for all.

In [4]:
QUERY = "slurry"           # e.g. "nvz", "ssafo", "" for everything
DOCUMENT_TYPE = None       # e.g. "guidance" / "detailed_guide" / None
HAS_BODY = None            # True / False / None

hit = df
if QUERY:
    q = QUERY.casefold()
    blob = (
        hit["title"].str.casefold()
        + " "
        + hit["description"].str.casefold()
        + " "
        + hit["url"].str.casefold()
        + " "
        + hit["document_type"].str.casefold()
    )
    hit = hit[blob.str.contains(q, regex=False)]
if DOCUMENT_TYPE:
    hit = hit[hit.document_type == DOCUMENT_TYPE]
if HAS_BODY is not None:
    hit = hit[hit.has_body == HAS_BODY]

view = hit.sort_values("view_count", ascending=False)[
    ["title", "document_type", "updated_at", "view_count", "body_len", "has_body", "url"]
].reset_index(drop=False)
print(f"{len(view):,} / {len(df):,} rows")
view

22 / 16,232 rows


,index,title,document_type,updated_at,view_count,body_len,has_body,url
0,261,Farming Equipment and Technology Fund (FETF) 2026 (Closed),guidance,2026-05-12 11:15:00+00:00,1980,1822,True,https://www.gov.uk/government/publications/farming-equipment-and-technology-fund-fetf-2026-closed
1,539,Farming Equipment and Technology Fund (FETF) 2025 (closed),guidance,2026-03-02 00:15:00+00:00,828,341,True,https://www.gov.uk/government/publications/farming-equipment-and-technology-fund-2025
2,682,Farming Equipment and Technology Fund (FETF) 2024 (closed),guidance,2025-04-09 15:58:47+00:00,607,399,True,https://www.gov.uk/government/publications/farming-equipment-and-technology-fund-2024
3,1372,"Storing silage, slurry and agricultural fuel oil",detailed_guide,2026-01-26 09:30:02+00:00,212,13976,True,https://www.gov.uk/guidance/storing-silage-slurry-and-agricultural-fuel-oil
4,1589,"RP28: Roofing (sprayer washdown area, manure storage area, livestock gathering area, slurry stores, silage stores)",countryside_stewardship_grant,2026-05-28 10:10:49+00:00,176,6680,True,https://www.gov.uk/countryside-stewardship-grants/roofing-sprayer-washdown-area-manure-storage-area-livestock-gather...
5,3154,"Construct a new or substantially enlarge an existing silage, slurry or agricultural fuel oil store, or store field s...",detailed_guide,2026-01-26 09:30:00+00:00,55,1263,True,https://www.gov.uk/guidance/construct-a-new-or-substantially-enlarge-an-existing-silage-slurry-or-agricultural-fuel-...
6,3677,AQ1: Automatic slurry scraper,countryside_stewardship_grant,2026-05-28 10:32:24+00:00,41,3965,True,https://www.gov.uk/countryside-stewardship-grants/aq1-automatic-slurry-scraper
7,3721,Slurry Infrastructure grant round 2: guidance for invited applicants,guidance,2026-04-16 14:45:29+00:00,40,1205,True,https://www.gov.uk/government/publications/slurry-infrastructure-grant-round-2-guidance-for-invited-applicants
8,4514,Preventing agricultural pollution in exceptional circumstances,detailed_guide,2022-08-04 15:10:29+00:00,27,7277,True,https://www.gov.uk/guidance/preventing-agricultural-pollution-in-exceptional-weather
9,4661,T24: anaerobic digestion on farms,detailed_guide,2025-07-01 08:30:02+00:00,25,4223,True,https://www.gov.uk/guidance/waste-exemption-t24-anaerobic-digestion-at-premises-used-for-agriculture-and-burning-res...


## Inspect one page

`inspect(i)` uses the **original `df` index**. Copy `index` from the filter table above, or pass a URL.

In [5]:
def inspect(key: int | str, body_chars: int = 4000) -> None:
    if isinstance(key, str) and key.startswith("http"):
        row = df.loc[df.url == key].iloc[0]
    else:
        row = df.loc[int(key)]
    body = row.body_text or ""
    clipped = body[:body_chars] + ("\n\n… [truncated]" if len(body) > body_chars else "")
    display(HTML(f"""
    <div style="font-family: ui-sans-serif, system-ui; line-height: 1.4">
      <h3 style="margin-bottom: 0.2rem">{row.title}</h3>
      <p><a href="{row.url}" target="_blank">{row.url}</a></p>
      <p style="color:#555">{row.description}</p>
      <table>
        <tr><th align="left">document_type</th><td>{row.document_type}</td></tr>
        <tr><th align="left">updated_at</th><td>{row.updated_at}</td></tr>
        <tr><th align="left">view_count</th><td>{row.view_count:,}</td></tr>
        <tr><th align="left">content_id</th><td><code>{row.content_id}</code></td></tr>
        <tr><th align="left">body_len</th><td>{row.body_len:,} chars &nbsp; has_body={row.has_body} &nbsp; in_with_body={row.in_with_body}</td></tr>
      </table>
      <pre style="white-space: pre-wrap; background:#111; color:#eee; padding: 1rem; max-height: 32rem; overflow: auto">{clipped}</pre>
    </div>
    """))


# first filter hit, or row 0 if the filter is empty
inspect(int(view["index"].iloc[0]) if len(view) else 0)

document_type,guidance
updated_at,2026-05-12 11:15:00+00:00
view_count,"1,980"
content_id,8e2c6659-9470-475e-9873-e22e1c70aa55
body_len,"1,822 chars has_body=True in_with_body=True"


## Live search (widgets)

Skip this cell if you don't have `ipywidgets`. The `QUERY` cell above is enough.

In [6]:
try:
    import ipywidgets as w

    query_box = w.Text(value="slurry", description="query", continuous_update=False)
    type_box = w.Dropdown(
        options=["(all)"] + sorted(df.document_type.unique().tolist()),
        description="type",
    )
    n_box = w.IntSlider(value=25, min=5, max=200, step=5, description="rows")
    pick = w.IntText(value=0, description="inspect idx")
    go = w.Button(description="inspect")
    table_out = w.Output()
    page_out = w.Output()

    def filtered() -> pd.DataFrame:
        sub = df
        q = query_box.value.strip().casefold()
        if q:
            blob = (
                sub.title.str.casefold()
                + " "
                + sub.description.str.casefold()
                + " "
                + sub.url.str.casefold()
            )
            sub = sub[blob.str.contains(q, regex=False)]
        if type_box.value != "(all)":
            sub = sub[sub.document_type == type_box.value]
        return sub.sort_values("view_count", ascending=False)

    def render_table(_=None):
        table_out.clear_output()
        sub = filtered()
        with table_out:
            print(f"{len(sub):,} / {len(df):,}")
            display(
                sub.head(n_box.value)[
                    ["title", "document_type", "updated_at", "view_count", "body_len", "url"]
                ]
            )

    def render_page(_=None):
        page_out.clear_output()
        sub = filtered()
        with page_out:
            if sub.empty:
                print("no rows")
                return
            idx = sub.index[min(pick.value, len(sub) - 1)]
            inspect(int(idx))

    for ctrl in (query_box, type_box, n_box):
        ctrl.observe(render_table, names="value")
    go.on_click(render_page)
    render_table()
    display(w.VBox([w.HBox([query_box, type_box, n_box]), table_out, w.HBox([pick, go]), page_out]))
except ImportError:
    print("ipywidgets not installed — use the QUERY cell")